In [ ]:
import pandas as pd

In [ ]:
business_df = pd.read_json('dataset/yelp_dataset/yelp_academic_dataset_business.json', lines=True)
len(business_df)

In [ ]:
# Cleaning yelp_academic_dataset_business.json file
# Filtered file to only show businesses with more than 20 reviews

business_df = business_df[business_df['review_count'] >=20]
len(business_df)

In [ ]:
business_df.columns

In [ ]:
# Testing to see if there is a correlation between avg ratings
# and what state the restraunt is in

state_average = business_df.groupby('state')['stars'].mean()
print(state_average)

In [ ]:
# Removed unnecessary columns
business_df.drop(columns=['address', 'city', 'postal_code', 'latitude', 'longitude', 'is_open', 'hours', 'attributes', 'categories', 'name'], inplace=True)
business_df.columns

In [ ]:
# Renaming the 'stars' feature to 'business_star'
business_df.rename(columns={'stars': 'business_star'}, inplace=True)
business_df.columns

In [ ]:
# The file was too big for my laptop to handle
# Loaded the files in chunks, dropped unnecessary 
# columns and wrote it out to memory

file_in = 'dataset/yelp_dataset/yelp_academic_dataset_review.json'
file_out = 'dataset/cleaned_dataset/cleaned_review.json'

chunk_size = 200000

with open(file_out, 'w') as out:
    first_chunk = True

    for chunk in pd.read_json(file_in, lines=True, chunksize=chunk_size):
        chunk.drop(columns=['user_id', 'useful', 'funny', 'cool', 'date'], inplace=True)
        chunk.dropna(inplace=True)

        if first_chunk:
            chunk.to_json(out, orient='records', lines=True)
        else:
            chunk.to_json(out, orient='records', lines=True, mode='a')

In [ ]:
review_df = pd.read_json('dataset/cleaned_dataset/cleaned_review.json', lines=True)
review_df.columns

In [ ]:
business_df.columns

In [ ]:
# finding the average of all review ratings and merging it to bussiness

average_review_ratings_df = review_df[['business_id', 'stars']]
average_review_ratings_df = average_review_ratings_df.groupby('business_id', as_index=False).agg(average_review_ratings_df=('stars', 'mean'))

average_review_ratings_df.head()

In [ ]:
business_df = business_df.merge(average_review_ratings_df, on='business_id', how='inner')
business_df.head

In [ ]:
review_df.columns

In [ ]:
review_df

In [ ]:
# Dropping the review_id and y_label

review_df.drop(columns=['review_id', 'stars'], inplace=True)
review_df.columns


In [ ]:
# concatenating all the reviews onto one column and merging
# it with the main df

review_df = review_df.groupby('business_id', as_index=False).agg(text=('text', '\n'.join))
review_df.head

In [ ]:
business_df = business_df.merge(review_df, on='business_id', how='inner')
business_df.columns

In [ ]:
business_df.head()

In [ ]:
business_df.shape

In [ ]:
# Renaming feature columns

business_df.rename({'average_review_ratings_df': 'avg_review_ratings', 'text': 'reviews'}, inplace=True)
business_df.columns

In [ ]:
business_df.head

In [ ]:
# writing out the main dataframe to memory

chunk_size = 200000
file_out = 'dataset/cleaned_dataset/training_dataset.json'

with open(file_out, 'w') as out:
    for i in range(0, len(business_df), chunk_size):
        chunk = business_df.iloc[i:i+chunk_size]
        chunk.to_json(out, orient='records', lines=True)
